In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import load_iris
from sklearn.metrics import f1_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

from students.kudryavcev.lesson2 import Exercise

In [ ]:
iris = load_iris()
X = iris.data.astype(float)
y = iris.target

y_binary = (y == 0).astype(int)

print(f"Размер датасета: {X.shape}")
print(f"Класс 1 (positive): {int(np.sum(y_binary == 1))}")
print(f"Класс 0 (negative): {int(np.sum(y_binary == 0))}")

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y_binary, test_size=0.6, random_state=42, stratify=y_binary)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Тренировочная выборка: {len(X_train)} объектов")
print(f"Валидационная выборка: {len(X_val)} объектов")
print(f"Тестовая выборка: {len(X_test)} объектов")

In [ ]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1.0

X_train_norm = (X_train - mean) / std
X_val_norm = (X_val - mean) / std
X_test_norm = (X_test - mean) / std

In [ ]:
def evaluate_model(lr, batch_size, n_iter, seed):
    model = Exercise.create_logistic_model(num_features=X_train.shape[1], rng=np.random.default_rng(seed))

    Exercise.fit(model, X_train_norm, y_train, lr=lr, n_iter=n_iter, batch_size=batch_size)

    y_pred_proba = model.predict(X_val_norm)
    y_pred = (y_pred_proba >= 0.5).astype(int)

    accuracy = float(np.mean(y_pred == y_val))
    auroc = float(roc_auc_score(y_val, y_pred_proba))

    recall = float(recall_score(y_val, y_pred, zero_division=0))
    f1 = float(f1_score(y_val, y_pred, zero_division=0))

    return accuracy, auroc, recall, f1

In [ ]:
learning_rates = [0.001, 0.005, 0.01, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.5]
batch_sizes = [1, 2, 4, 8, 16, 32, None]
epochs_list = [1, 2, 3, 4, 5, 10, 15]
seeds = [42, 123, 777]

results = []

for lr in learning_rates:
    for bs in batch_sizes:
        for n_iter in epochs_list:
            accuracy_scores = []
            auroc_scores = []
            recall_scores = []
            f1_scores = []

            for seed in seeds:
                acc, auc, rec, f1 = evaluate_model(lr, bs, n_iter, seed)
                accuracy_scores.append(acc)
                auroc_scores.append(auc)
                recall_scores.append(rec)
                f1_scores.append(f1)

            results.append(
                {
                    "lr": lr,
                    "batch_size": bs if bs is not None else "Full batch",
                    "n_iter": n_iter,
                    "mean_accuracy": np.mean(accuracy_scores),
                    "mean_auroc": np.mean(auroc_scores),
                    "mean_recall": np.mean(recall_scores),
                    "mean_f1": np.mean(f1_scores),
                }
            )

df_results = pd.DataFrame(results)

df_results = df_results.sort_values(by=["mean_f1", "mean_auroc", "n_iter"], ascending=[False, False, True]).reset_index(
    drop=True
)

display(df_results.head(10))

Что если данных будет меньше:

In [ ]:
tiny_X_train = X_train_norm[:10]
tiny_y_train = y_train[:10]

print(f"Обучаемся на {len(tiny_X_train)} объектах")

starved_model = Exercise.create_logistic_model(num_features=tiny_X_train.shape[1], rng=np.random.default_rng(42))

Exercise.fit(starved_model, tiny_X_train, tiny_y_train, lr=0.08, n_iter=1, batch_size=1)

starved_prob = starved_model.predict(X_val_norm)
starved_pred = (starved_prob >= 0.5).astype(int)

print(f"Accuracy: {float(np.mean(starved_pred == y_val)):.4f}")
print(f"AUROC:    {float(roc_auc_score(y_val, starved_prob)):.4f}")
print(f"F1:       {float(f1_score(y_val, starved_pred, zero_division=0)):.4f}")

In [ ]:
best_lr = 0.08
best_bs = 1
best_iter = 1

print("Итоговые гиперпараметры для Exercise.get_iris_hyperparameters():")
print(f"lr = {best_lr}")
print(f"batch_size = {best_bs}")
print(f"n_iter = {best_iter}")

X_trainval = np.vstack([X_train_norm, X_val_norm])
y_trainval = np.concatenate([y_train, y_val])

final_model = Exercise.create_logistic_model(num_features=X_trainval.shape[1], rng=np.random.default_rng(42))

Exercise.fit(final_model, X_trainval, y_trainval, lr=best_lr, n_iter=best_iter, batch_size=best_bs)

y_test_prob = final_model.predict(X_test_norm)

y_test_pred = (y_test_prob >= 0.5).astype(int)

test_accuracy = float(np.mean(y_test_pred == y_test))
test_auc = float(roc_auc_score(y_test, y_test_prob))
test_recall = float(recall_score(y_test, y_test_pred, zero_division=0))
test_f1 = float(f1_score(y_test, y_test_pred, zero_division=0))

print("\nФинальные метрики на тестовой выборке:")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"AUROC:    {test_auc:.4f}")
print(f"Recall:   {test_recall:.4f}")
print(f"F1-score: {test_f1:.4f}")